# Bank Loan Approval System

1. Import libraries

In [14]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib

2. Load the dataset

In [15]:
df = pd.read_csv("/content/Bank_Loan_Portfolio_Risk_Analytics.csv")

df.head()

,Loan_ID,Customer_ID,Age,Gender,Marital_Status,Dependents,Education,Employment_Type,Annual_Income,Monthly_Income,...,Loan_Term,Interest_Rate,Debt_to_Income_Ratio,Collateral_Value,Application_Date,Loan_Region,Approval_Status,Disbursement_Status,Default_Status,Repayment_Status
0,L00001,C00001,59,Male,Single,0,Graduate,Salaried,6632961,413227,...,48,15.08,0.25,11356782,2023-01-01,North,Rejected,Not Disbursed,Default,Late
1,L00002,C00002,49,Male,Single,4,Graduate,Self Employed,7742510,399842,...,36,11.69,0.43,63235849,2023-01-02,North,Approved,Disbursed,No Default,On Time
2,L00003,C00003,35,Male,Married,4,Graduate,Self Employed,7702731,398704,...,60,7.09,0.30,68477104,2023-01-03,North,Approved,Disbursed,No Default,On Time
3,L00004,C00004,63,Female,Married,0,Not Graduate,Salaried,1244461,419051,...,36,17.56,0.70,5771712,2023-01-04,North,Rejected,Not Disbursed,Default,Late
4,L00005,C00005,28,Female,Married,4,Not Graduate,Business,1767256,305790,...,120,11.40,0.41,11737018,2023-01-05,West,Rejected,Not Disbursed,Default,Late


3. Data Cleaning

In [16]:
df.drop(
    [
        "Loan_ID",
        "Customer_ID",
        "Application_Date"
    ],
    axis=1,
    inplace=True
)

In [17]:
df.isnull().sum()

,0
Age,0
Gender,0
Marital_Status,0
Dependents,0
Education,0
Employment_Type,0
Annual_Income,0
Monthly_Income,0
Credit_Score,0
Existing_Loan_Count,0


4. Convert Target Variable


In [18]:
df["Default_Status"] = df["Default_Status"].map(
    {
        "Default":1,
        "No Default":0
    }
)

5. Split Features and Target

In [19]:
X = df.drop(
    "Default_Status",
    axis=1
)


y = df["Default_Status"]


noise_index = np.random.choice(
    y.index,
    size=int(len(y)*0.08),
    replace=False
)


y.loc[noise_index] = 1 - y.loc[noise_index]

/tmp/ipykernel_1981/3492216805.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  y.loc[noise_index] = 1 - y.loc[noise_index]


6. Identify Columns

In [20]:
categorical_cols = X.select_dtypes(
    include="object"
).columns


numeric_cols = X.select_dtypes(
    exclude="object"
).columns


print(categorical_cols)
print(numeric_cols)

Index(['Gender', 'Marital_Status', 'Education', 'Employment_Type', 'Loan_Type',
       'Loan_Region', 'Approval_Status', 'Disbursement_Status',
       'Repayment_Status'],
      dtype='object')
Index(['Age', 'Dependents', 'Annual_Income', 'Monthly_Income', 'Credit_Score',
       'Existing_Loan_Count', 'Loan_Amount', 'Loan_Term', 'Interest_Rate',
       'Debt_to_Income_Ratio', 'Collateral_Value'],
      dtype='object')


7. Create PreProcessing Pipeline

In [21]:
preprocessor = ColumnTransformer(

    transformers=[

        (
            "num",
            StandardScaler(),
            numeric_cols
        ),


        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_cols
        )

    ]
)

8. Train-Test Split

In [22]:
X_train, X_test, y_train, y_test = train_test_split(

    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y

)

9. Random Forest Model

In [23]:
model = RandomForestClassifier(

    n_estimators=200,
    max_depth=8,
    random_state=42

)

10. Create ML Pipeline

In [24]:
pipeline = Pipeline(

    steps=[

        (
            "preprocessor",
            preprocessor
        ),

        (
            "model",
            model
        )

    ]

)

11. Perform Model Training

In [25]:
pipeline.fit(
    X_train,
    y_train
)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['Age', 'Dependents', 'Annual_Income', 'Monthly_Income', 'Credit_Score',
       'Existing_Loan_Count', 'Loan_Amount', 'Loan_Term', 'Interest_Rate',
       'Debt_to_Income_Ratio', 'Collateral_Value'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['Gender', 'Marital_Status', 'Education', 'Employment_Type', 'Loan_Type',
       'Loan_Region', 'Approval_Status', 'Disbursement_Status',
       'Repayment_Status'],
      dtype='object'))])),
                ('model',
                 RandomForestClassifier(max_depth=8, n_estimators=200,
                                        random_state=42))])

12. Evaluate the model

In [26]:
prediction = pipeline.predict(
    X_test
)


print(
accuracy_score(
    y_test,
    prediction
)
)


print(
classification_report(
    y_test,
    prediction
)
)

0.905
              precision    recall  f1-score   support

           0       0.89      0.88      0.88        83
           1       0.92      0.92      0.92       117

    accuracy                           0.91       200
   macro avg       0.90      0.90      0.90       200
weighted avg       0.90      0.91      0.90       200



13. Saving the model

In [27]:
import os

os.makedirs(
    "models",
    exist_ok=True
)

joblib.dump(

    pipeline,

    "models/loan_default_model.pkl"

)

print("Saved Successfully")

Saved Successfully


**14. Accuracy of the model ~ 90.5%**